### Group 24

Shiref Khaled Elhalawany -  221100944

Ahmed Anis Hassan - 221100101 

Karim Ashraf Elsayed - 221100391

Kareem Shaheen - 221101524

# Section 1: Utility Functions and Configuration

This notebook contains common imports, constants, file paths, and helper functions used across the Section 1 analysis notebooks.

### 1. Imports and Constants
Standard libraries for data manipulation, plotting, and system operations, along with project-wide constants.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

# Constants
RANDOM_STATE = 42
TARGET_MIN_USERS = 10000
TARGET_MIN_ITEMS = 500
TARGET_MIN_RATINGS = 100000

print("Libraries imported and constants defined.")

### 2. File Paths and Directory Setup
Defines input/output paths and ensures necessary directories exist.

In [ ]:
# Paths
tables_path = '../results/tables/'
plots_path = '../results/plots/'
RAW_DATA_PATH = '../data/ml-20m/ml-20m/ratings.csv'
SAMPLED_DATA_PATH = tables_path + 'sampled_scaled_ratings.csv'

if not os.path.exists(tables_path):
    os.makedirs(plots_path)
if not os.path.exists(plots_path):
    os.makedirs(tables_path)

print(f"Paths defined:\nTables: {tables_path}\nPlots: {plots_path}")

### 3. Utility: Execution Time Logging
Helper to track and log the duration of various processing stages.

In [ ]:
runtime_rows = []

def log_time(method, stage, seconds, extra=""):
    """
    Logs execution time for a specific method and stage.
    """
    runtime_rows.append({
        "Method": method,
        "Stage": stage,
        "Seconds": seconds,
        "Extra": extra
    })
    print(f"Time logged for {method} - {stage}: {seconds:.4f}s")

### 4. Utility: Memory Usage Calculation
Helper to estimate the RAM usage of NumPy arrays and Pandas DataFrames.

In [ ]:
def memory_mb(*arrays):
    """
    Calculates the total memory usage of given numpy arrays or pandas objects in MB.
    """
    total_bytes = 0
    for arr in arrays:
        if hasattr(arr, "values"):
            total_bytes += arr.values.nbytes
        else:
            total_bytes += arr.nbytes
    return total_bytes / (1024 ** 2)

### 5. Utility: Error Metrics Calculation
Standardized calculation for Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).

In [ ]:
def calculate_metrics(true_values, pred_values):
    """
    Calculates Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).
    """
    if len(true_values) == 0 or len(pred_values) == 0:
        return np.nan, np.nan
        
    mae = mean_absolute_error(true_values, pred_values)
    rmse = np.sqrt(mean_squared_error(true_values, pred_values))
    return mae, rmse

### 6. Data Loading and Sampling
Logic to load raw ratings and sample a subset of users and items based on density constraints.

In [ ]:
def load_and_sample_data(raw_path, output_path):
    """
    Loads raw ratings data, samples it to meet target requirements, and returns the filtered DataFrame.
    """
    print(f"Loading raw data from {raw_path}...")
    df_raw = pd.read_csv(raw_path)
    df_raw.columns = ['UserID', 'ItemID', 'Rating', 'Timestamp']
    
    np.random.seed(RANDOM_STATE)
    
    all_items = df_raw['ItemID'].unique()
    sampled_items = np.random.choice(all_items, size=TARGET_MIN_ITEMS, replace=False) if len(all_items) > TARGET_MIN_ITEMS else all_items
    df_items_subset = df_raw[df_raw['ItemID'].isin(sampled_items)]
    
    selected_users = set()
    
    for item in sampled_items:
        users_who_rated = df_items_subset[df_items_subset['ItemID'] == item]['UserID'].unique()
        if len(users_who_rated) > 0:
            selected_users.add(np.random.choice(users_who_rated))
            
    available_users = df_items_subset['UserID'].unique()
    np.random.shuffle(available_users)
    
    current_ratings = df_items_subset[df_items_subset['UserID'].isin(selected_users)].shape[0]
    
    chunk_size = 2000 
    for i in range(0, len(available_users), chunk_size):
        if len(selected_users) >= TARGET_MIN_USERS and current_ratings >= TARGET_MIN_RATINGS:
            break
            
        chunk = available_users[i : i + chunk_size]
        selected_users.update(chunk)
        
        current_ratings = df_items_subset[df_items_subset['UserID'].isin(selected_users)].shape[0]
   
    df_final = df_items_subset[df_items_subset['UserID'].isin(selected_users)].copy()
    
    return df_final

### 7. Visualization: Scree and Cumulative Variance Plots
Function to visualize the explained variance of principal components.

In [ ]:
def plot_explained_variance(singular_values, plots_path, filename_prefix, threshold=0.90):
    """
    Generates and saves Scree Plot and Cumulative Explained Variance Plot.
    """
    if singular_values is None or len(singular_values) == 0:
        print("No singular values provided for plotting.")
        return

    # Scree Plot
    plt.figure(figsize=(10, 6))
    plt.plot(singular_values, marker='o', linestyle='--')
    plt.title('Scree Plot: Singular Values')
    plt.xlabel('Index')
    plt.ylabel('Singular Value')
    plt.grid(True)
    scree_path = f"{plots_path}{filename_prefix}_scree_plot.png"
    plt.savefig(scree_path)
    plt.show()
    print(f"Scree plot saved to {scree_path}")

    # Cumulative Explained Variance
    total_variance = np.sum(singular_values**2)
    explained_variance = (singular_values**2) / total_variance
    cumulative_variance = np.cumsum(explained_variance)

    plt.figure(figsize=(10, 6))
    plt.plot(cumulative_variance, marker='o', linestyle='-', color='orange')
    plt.axhline(y=threshold, color='r', linestyle='--', label=f'{int(threshold*100)}% Explained Variance')
    plt.title('Cumulative Explained Variance')
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Variance')
    plt.legend()
    plt.grid(True)
    cum_path = f"{plots_path}{filename_prefix}_cumulative_variance.png"
    plt.savefig(cum_path)
    plt.show()
    print(f"Cumulative Variance plot saved to {cum_path}")

### 8. Analysis: MLE Covariance Calculation
Computes Item-Item covariance matrix while correctly handling missing values (NaNs) by using only overlapping ratings.

In [ ]:
def compute_mle_covariance(user_item_matrix_centered):
    """
    Computes the Maximum Likelihood Estimation (MLE) covariance matrix,
    handling NaNs by only considering overlapping ratings between item pairs.
    """
    X = user_item_matrix_centered 
    items = X.columns
    m = len(items)
    Xv = X.values 
    cov_mle = np.zeros((m, m), dtype=float)

    print(f"Computing MLE Covariance Matrix for {m} items...")
    for i in range(m):
        xi = Xv[:, i]
        for j in range(i, m):
            xj = Xv[:, j]
            # Only use users who rated both items
            mask = ~np.isnan(xi) & ~np.isnan(xj)
            n = int(mask.sum())

            if n == 0:
                cij = 0.0
            else:
                cij = float(np.dot(xi[mask], xj[mask]) / n)

            cov_mle[i, j] = cij
            cov_mle[j, i] = cij
            
    cov_df = pd.DataFrame(cov_mle, index=items, columns=items)
    return cov_df

### 9. Analysis: Latent Peer Identification
Identifies top similar items (peers) in the reduced latent feature space.

In [ ]:
def find_latent_peers(n_components, top_n_peers, item_ids, target_ids, evecs):
    """
    Identifies top N peer items based on cosine similarity in the latent feature space.
    """
    latent_features = evecs[:, :n_components]  
    latent_df = pd.DataFrame(latent_features, index=item_ids)

    peers_result = {}
    for tid in target_ids:
        if tid not in latent_df.index:
            continue

        target_vec = latent_df.loc[tid].values.reshape(1, -1)
        sim_scores = cosine_similarity(target_vec, latent_df.values).flatten()
        sim_series = pd.Series(sim_scores, index=latent_df.index).drop(tid)
        sorted_sim = sim_series.sort_values(ascending=False)

        peers_result[tid] = {
            'top_ids': sorted_sim.head(top_n_peers).index.tolist(),
            'sim_series': sim_series
        }
    return peers_result